# WebSky Data Analysis
---
CMB + tSZ + noise
---
---

In [2]:
### IMPORTS ###

import healpy as hp
import numpy as np
import pandas as pd
import scipy as sp
import numba as nb
import matplotlib.pyplot as plt

In [3]:
### MEMORY MANAGEMENT ###

import gc # Garbage Collector

# Before starting a new run, clear previous big variables if they exist
if 'data_cube' in locals():
    del data_cube
if 'alms_list' in locals():
    del alms_list
if 'y_hat_harmonic' in locals():
    del y_hat_harmonic

gc.collect() # Manually trigger memory cleanup

1419

In [ ]:
### 1. Load WebSky Components --- https://lambda.gsfc.nasa.gov/simulation/mocks_data.html --- CMB in muK units ###

# These are usually at high resolution (Nside 4096). 
# You might need to downgrade them to Nside 128 or 256 for faster testing.
path_to_websky = '/Users/joanribot/HEAVY_STUFF/TFM_data/WebSky_CMB_Mocks/'

# Set your working resolution and corresponding lmax
nside_work = 2048
#nside_work = 256

# A. Load tSZ, downgrade and save it
y_map_raw = hp.read_map(path_to_websky + "tsz.fits")
y_map = hp.ud_grade(y_map_raw, nside_out=nside_work)
#hp.write_map(path_to_websky + "tsz_downgraded.fits", y_map)
hp.write_map(path_to_websky + "websky_truth_y_map.fits", y_map, coord='G', overwrite=True, dtype=np.float32)

# B. Load CMB alms and let it detect the original Lmax ######### CHECK UNITS FOR CMB TOO
# Use return_mmax=True to be extra safe with WebSky's format
alms_cmb, mmax_in = hp.read_alm(path_to_websky + "unlensed_alm.fits", hdu=1, return_mmax=True)

# C. Calculate the Lmax of the file data
lmax_in = hp.Alm.getlmax(len(alms_cmb), mmax_in)

# D. Convert to map. 
# We tell it the ALMs are at 'lmax_in', but we want the result at 'nside_work'
cmb_map = hp.alm2map(alms_cmb, nside=nside_work, lmax=lmax_in, mmax=mmax_in)

# E. Save the clean CMB map (in muK)
# This is your 'Ground Truth' for the CILC (which tries to remove this)
hp.write_map(path_to_websky + "websky_truth_cmb_map.fits", cmb_map, coord='G', overwrite=True, dtype=np.float32)